# ML - Final Project
## Classification
**Project:** Multilingual Language Identification System
**Group Members:** 
1. Seyyed Sina Alizadeh Tabatabai - 810101477
2. Majid Sadeghinejad - 810101459
3. Matin Shiasi - 810101453

## 1. Introduction
The objective of this notebook is to train machine learning models (SVM, Random Forest, MLP) to classify audio samples into four languages: German, Italian, Korean, and Spanish.

We utilize a Grouped Stratified Split strategy to ensure robust evaluation. Since the dataset consists of audio clips uploaded by specific students (where one student likely uploaded multiple clips from the same speaker/source), we must ensure that all clips from a single student remain together. This prevents "Speaker Leakage," where the model memorizes a specific voice rather than learning language features.

In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

DATA_FILE = '../Output/CSV/extracted_features.csv'
RANDOM_STATE = 42

print("Libraries imported.")


Libraries imported.


## 2. Data Loading and Splitting (Grouped by Source)

**Handling Data Leakage:**
Our filenames follow the structure: `{StudentID}_{Gender}_{Language}_{ClipID}.mp3` .
To prevent the model from memorizing specific speakers or recording sessions, we group the data by `{StudentID}`. This ensures that if a student's data is used for training, none of their clips appear in the test set.

*   Group: Extracted from the first segment of the filename.
*   Split Strategy: We use GroupShuffleSplit to create an 80/20 split based on these unique Student IDs.


In [4]:
df = pd.read_csv(DATA_FILE)

df['student_id'] = df['filename'].apply(lambda x: x.replace('noise_', '').split('_')[0])

print(f"Unique Student IDs found: {df['student_id'].nunique()}")


X = df.drop(columns=['filename', 'label', 'gender', 'type', 'student_id'])
y = df['label']
groups = df['student_id']


le = LabelEncoder()
y_encoded = le.fit_transform(y)

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y_encoded, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y_encoded[train_idx]
y_test = y_encoded[test_idx]

print(f"Training Set: {X_train.shape[0]} samples")
print(f"Testing Set:  {X_test.shape[0]} samples")


train_ids = set(groups.iloc[train_idx])
test_ids = set(groups.iloc[test_idx])
print(f"Intersection of Student IDs (should be 0): {len(train_ids.intersection(test_ids))}")


Unique Student IDs found: 78
Training Set: 1142 samples
Testing Set:  298 samples
Intersection of Student IDs (should be 0): 0


## 3. Feature Scaling
We apply StandardScaler to normalize the features (mean=0, variance=1). This is critical for the SVM and MLP models to converge efficiently and weigh features equally.


In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaled.")


Data scaled.


## 4. Model Training

We train three classifiers with standard hyperparameters:
1.  SVM (RBF Kernel): Effective for high-dimensional, non-linear feature spaces.
2.  Random Forest: An ensemble of decision trees, providing robustness and feature importance.
3.  MLP (Neural Network): A deep learning approach to capture complex feature interactions.


In [53]:
svm_model = SVC(kernel='rbf', random_state=RANDOM_STATE)
svm_model.fit(X_train_scaled, y_train)
print("SVM Trained.")

rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_model.fit(X_train_scaled, y_train)
print("Random Forest Trained.")

mlp_model = MLPClassifier(hidden_layer_sizes=(1000, 1000, 1000, 1000), max_iter=int(1e5), random_state=RANDOM_STATE)
mlp_model.fit(X_train_scaled, y_train)
print("MLP Trained.")

SVM Trained.
Random Forest Trained.
MLP Trained.


## 5. Saving Models
We verify the preliminary accuracy and save the trained models for the final evaluation notebook.

Note: High accuracy here is expected if the number of unique speakers per language is low, as the models may distinguish languages based on speaker-specific voice characteristics.


In [56]:
print(f"SVM Accuracy: {accuracy_score(y_test, svm_model.predict(X_test_scaled)):.4f}")
print(f"RF Accuracy:  {accuracy_score(y_test, rf_model.predict(X_test_scaled)):.4f}")
print(f"MLP Accuracy: {accuracy_score(y_test, mlp_model.predict(X_test_scaled)):.4f}")

output_path = '../Output/PKL/'
joblib.dump(svm_model, output_path + 'svm_model.pkl')
joblib.dump(rf_model, output_path + 'rf_model.pkl')
joblib.dump(mlp_model, output_path + 'mlp_model.pkl')
joblib.dump(scaler, output_path + 'scaler.pkl')
joblib.dump(le, output_path + 'label_encoder.pkl')

print("Models saved.")


SVM Accuracy: 0.8322
RF Accuracy:  0.7416
MLP Accuracy: 0.8691
Models saved.
